<a href="https://colab.research.google.com/github/leahmaryann/debt_collection_classifier/blob/main/Debt_Collection_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Install libraries

In [1]:
!pip install datasets
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.7 MB/s eta 0:00:00


In [2]:
# Data handling
import pandas as pd
import numpy as np

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.layers import Dropout

# NLP preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ML tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# Hyperparameter tuning
import keras_tuner as kt

# Test cleaning
import re

###Creating the Dataset

In [3]:
# Created a labelled dataset for classification
# Each text represents a customer message
# Each label represents a category

data = {
"text":[

# willing_to_pay
"I will pay tomorrow",
"I can make a payment next week",
"I will settle the balance soon",
"I can pay on Friday",
"I will clear the account tomorrow",
"I will pay the outstanding balance",
"I will send the payment today",
"I can make payment this weekend",
"I will transfer the money tomorrow",
"I will settle the bill shortly",
"I will make the payment this week",
"I can pay the balance next month",

# financial_hardship
"I lost my job and cannot pay",
"I can't afford the payment right now",
"I am unemployed at the moment",
"I am struggling financially",
"I do not have money right now",
"I cannot make a payment this month",
"I am facing financial difficulties",
"I lost my income recently",
"I cannot afford to pay this debt",
"I my salary was reduced",
"I am going through financial hardship",
"I have no money at the moment",

# dispute
"This debt is not mine",
"I never opened this account",
"This account does not belong to me",
"I do not recognize this debt",
"I think this is a mistake",
"I did not apply for this account",
"I believe this account is incorrect",
"I never agreed to this debt",
"This charge is incorrect",
"I dispute this balance",
"I was never informed about this account",
"I want to dispute this account",

# refusal
"Please stop calling me",
"Stop contacting me immediately",
"Do not contact me again",
"I do not want any more calls",
"Stop sending messages",
"I refuse to discuss this debt",
"I will not pay this account",
"Leave me alone",
"Stop contacting me about this",
"Do not call me anymore",
"I do not want to talk about this debt",
"Remove my number from your system",

# payment_plan
"Can I pay in installments",
"I want to set up a payment plan",
"Can we arrange installments",
"I need a payment arrangement",
"I want to pay in smaller payments",
"Can we create a payment plan",
"I would like to pay monthly",
"I want to make partial payments",
"Can we spread the payments",
"I need to arrange a payment schedule",
"I want to pay over time",
"Can I split the payments"

],

"label":[

# willing_to_pay
"willing_to_pay","willing_to_pay","willing_to_pay","willing_to_pay",
"willing_to_pay","willing_to_pay","willing_to_pay","willing_to_pay",
"willing_to_pay","willing_to_pay","willing_to_pay","willing_to_pay",

# financial_hardship
"financial_hardship","financial_hardship","financial_hardship","financial_hardship",
"financial_hardship","financial_hardship","financial_hardship","financial_hardship",
"financial_hardship","financial_hardship","financial_hardship","financial_hardship",

# dispute
"dispute","dispute","dispute","dispute",
"dispute","dispute","dispute","dispute",
"dispute","dispute","dispute","dispute",

# refusal
"refusal","refusal","refusal","refusal",
"refusal","refusal","refusal","refusal",
"refusal","refusal","refusal","refusal",

# payment_plan
"payment_plan","payment_plan","payment_plan","payment_plan",
"payment_plan","payment_plan","payment_plan","payment_plan",
"payment_plan","payment_plan","payment_plan","payment_plan"

]
}


In [4]:
# Convert dictionary to Dataframe
df = pd.DataFrame(data)
df.head()

,text,label
0,I will pay tomorrow,willing_to_pay
1,I can make a payment next week,willing_to_pay
2,I will settle the balance soon,willing_to_pay
3,I can pay on Friday,willing_to_pay
4,I will clear the account tomorrow,willing_to_pay


Text Cleaning


In [5]:
# Clean and normalize the data

# Convert to lowercase
df["text"] = df["text"].str.lower()

# Remove punctuation
df["text"] = df["text"].apply(lambda x: re.sub(r'[^\w\s]', '', x))

# Remove extra spaces
df["text"] = df["text"].apply(lambda x: re.sub(r'\s+', ' ', x))

### Encode Labels

In [6]:
# Convert categorical labels into numeric values
encoder = LabelEncoder()

df["label_encoder"] = encoder.fit_transform(df["label"])
df


,text,label,label_encoder
0,i will pay tomorrow,willing_to_pay,4
1,i can make a payment next week,willing_to_pay,4
2,i will settle the balance soon,willing_to_pay,4
3,i can pay on friday,willing_to_pay,4
4,i will clear the account tomorrow,willing_to_pay,4
5,i will pay the outstanding balance,willing_to_pay,4
6,i will send the payment today,willing_to_pay,4
7,i can make payment this weekend,willing_to_pay,4
8,i will transfer the money tomorrow,willing_to_pay,4
9,i will settle the bill shortly,willing_to_pay,4


### Tokenisation & Padding

In [7]:
# Convert text into sequences of integers

# Limit vocabulary to 10000 words
tokenizer = Tokenizer(num_words=5000)

# Learn the word index
tokenizer.fit_on_texts(df["text"])

# Convert text to sequence
sequences = tokenizer.texts_to_sequences(df["text"])

# Pad sequences - to ensure sequences are equal length
X = pad_sequences(sequences, maxlen=20)

# Target variable
y = df["label_encoder"]

### Train-Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Model Construction for Hyperparameter Optimization

In [9]:
def build_model(hp):
  """
  Builds and compiles an LSTM-based text classification model.
    This function is used by Keras Tuner to search for the best
    hyperparameter configuration.

    The model includes:
    - An Embedding layer to convert words into dense vectors
    - An LSTM layer to capture sequence context
    - A Dropout layer to reduce overfitting
    - Dense layers for classification

    Args:
        hp (HyperParameters): Keras Tuner object used to define
                             and sample hyperparameters such as
                             embedding dimensions and LSTM units.

    Returns:
        model (tf.keras.Model): A compiled Keras model ready for training.
    """

  model = Sequential()

# Embedding layer:
# Transforms each word into a dense vector representation.
# The output dimension is tuned to find the best representation size.
  model.add(
      Embedding(
          input_dim=5000,
          output_dim=hp.Choice('embedding_dim', [32,64,128]),
          input_length=20
      )
  )

  # LSTM Layer:
  # Learns context and sequence patterns from the text.
  # Number of units is tuned for optimal performance.
  model.add(LSTM(hp.Choice('lstm_units', [32, 64, 128])))

  # Dropout layer:
  # Reduces overfitting by randomly disabling neurons during training.
  model.add(Dropout(0.3))

  # Hidden layer
  model.add(Dense(32, activation='relu'))

  # Output layer:
  # Uses softmax to classify into 5 categories.
  model.add(Dense(5, activation='softmax'))


  # Compile Model:
  model.compile(
      optimizer="adam",   # Adam optimizer for efficient training
      loss="sparse_categorical_crossentropy", # For multi-class classification
      metrics=["accuracy"]
  )

  return model

### Hyperparameter Tuning (Keras Tuner)
In this section, we define an LSTM-based neural network for intent classification.
We use Keras Tuner to automatically search for the best hyperparameters,
including embedding size and LSTM units.

In [10]:
# Random Search tuner to find the best model configuration.
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    executions_per_trial=2,
    directory="tuning",
    project_name="debt_nlp_v4"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [11]:
# Search for best hyperparameters
tuner.search(
    X_train,
    y_train,
    epochs=20,
    validation_data=(X_test,y_test)
)

Trial 5 Complete [00h 00m 15s]
val_accuracy: 0.6666666567325592

Best val_accuracy So Far: 0.6666666567325592
Total elapsed time: 00h 01m 16s


In [12]:
# Retrieve best performing model
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 128)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 775,877 (2.96 MB)

 Trainable params: 775,877 (2.96 MB)

 Non-trainable params: 0 (0.00 B)

### Train Best Model

In [13]:
best_model.fit(
    X_train,
    y_train,
    epochs=20,
    validation_data=(X_test, y_test)
)

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 421ms/step - accuracy: 0.7708 - loss: 1.5400 - val_accuracy: 0.7500 - val_loss: 1.5457
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.7708 - loss: 1.5107 - val_accuracy: 0.5833 - val_loss: 1.5362
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.8750 - loss: 1.4720 - val_accuracy: 0.7500 - val_loss: 1.5164
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.9167 - loss: 1.4340 - val_accuracy: 0.5833 - val_loss: 1.4898
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.7708 - loss: 1.3932 - val_accuracy: 0.6667 - val_loss: 1.4494
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.8125 - loss: 1.3288 - val_accuracy: 0.6667 - val_loss: 1.4076
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.8125 - loss: 1.2552 - val_accuracy: 0.5000 - val_loss: 1.3734
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.7500 - loss: 1.1391 - val_accuracy: 0.3333 - val_loss: 1.3535

### Prediction Function

In [14]:
def predict_label(text):
  """
    Predicts the intent label of a given input text using the trained model.

    The function preprocesses the input text by converting it into a sequence
    of integers and applying padding to match the model's expected input shape.
    It then uses the trained model to generate a prediction and returns the
    corresponding label.

    Args:
        text (str): The input message to classify.

    Returns:
        str: The predicted intent label (e.g., 'willing_to_pay', 'dispute').
    """
  seq = tokenizer.texts_to_sequences([text])

  padded = pad_sequences(seq, maxlen=20)

  prediction = best_model.predict(padded)

  label = encoder.inverse_transform([np.argmax(prediction)])

  return label[0]

In [15]:
# Evaluate model performance using classification metrics

# Step 1: Predict classes on test data
y_pred_probs = best_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Step 2: Generate classification report
report = classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
)

# Step 3: Display results
print("Classification Report:\n")
print(report)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step
Classification Report:

                    precision    recall  f1-score   support

           dispute       0.33      1.00      0.50         1
financial_hardship       0.50      1.00      0.67         2
      payment_plan       1.00      0.75      0.86         4
           refusal       1.00      0.33      0.50         3
    willing_to_pay       1.00      0.50      0.67         2

          accuracy                           0.67        12
         macro avg       0.77      0.72      0.64        12
      weighted avg       0.86      0.67      0.67        12



### Routing Logic

In [16]:
# Routing Table -Define actions based on predicted categories
routing_table = {
    "willing_to_pay": "Send payment link",
    "financial_hardship": "Send to financial hardship team",
    "dispute": "Send to dispute team",
    "refusal": "Send to refusal team",
    "payment_plan": "Offer installment plan"
}

def route_message(label):
  """
  Routes a message based on its predicted label.

  Args:
      label (str): The predicted intent label.

  Returns:
      str: The corresponding route action.
  """
  if label in routing_table:
    return routing_table[label]
  else:
    return "No route found"

In [23]:
def predict_and_route_with_confidence(text, threshold=0.7):
    """
    Predicts the intent of a message and routes it.
    If the model is not confident, routes to a human agent.

    Args:
        text (str): The customer message
        threshold (float): Minimum probability to trust the model

    Returns:
        tuple: (predicted_label, route_action, confidence)
    """
    # Step 1: Convert text to sequence and pad it
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=20)

    # Step 2: Get prediction probabilities from the model
    pred_probs = best_model.predict(padded, verbose=0)

    # Step 3: Get the max probability and predicted label
    max_prob = np.max(pred_probs)
    label = encoder.inverse_transform([np.argmax(pred_probs)])[0]

    # Step 4: Check confidence threshold
    if max_prob < threshold:
        return "uncertain", "Send to human agent", max_prob

    # Step 5: Route using the routing table
    route = route_message(label)

    return label, route, max_prob

### Testing


In [24]:
# Test with sample messages
texts = [
    "I can pay the balance on Friday",
    "I lost my job and cannot pay",
    "Stop calling me",
    "This is totally wrong"
]

for text in texts:
    label, route, confidence = predict_and_route_with_confidence(text)
    print(f"Message: {text}")
    print(f"Predicted Label: {label}")
    print(f"Route: {route}")
    print(f"Confidence: {confidence:.2f}")
    print("-" * 40)

Message: I can pay the balance on Friday
Predicted Label: willing_to_pay
Route: Send payment link
Confidence: 0.95
----------------------------------------
Message: I lost my job and cannot pay
Predicted Label: financial_hardship
Route: Send to financial hardship team
Confidence: 0.87
----------------------------------------
Message: Stop calling me
Predicted Label: uncertain
Route: Send to human agent
Confidence: 0.59
----------------------------------------
Message: This is totally wrong
Predicted Label: dispute
Route: Send to dispute team
Confidence: 0.84
----------------------------------------
